# Indici full-text dei nomi

Sei indici Lucene in Neo4j, con l'analizzatore `standard-folding` (minuscole, senza accenti): giocatori, federazioni (nome del paese e nome ufficiale), club, stadi (nome e città), allenatori, arbitri.

In [ ]:
# [Cella 1] Setup: codice del retrieval (src/wc_retrieval) e connessione a Neo4j
# importlib.reload ricarica il codice di src/ dopo una modifica, senza riavviare il kernel
import importlib
import sys
from getpass import getpass
from pathlib import Path

# Radice del progetto e cartella del codice
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
assert (SRC / "wc_retrieval" / "ricerca.py").exists(), f"Non trovo il codice in {SRC}"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import wc_retrieval.grafo
import wc_retrieval.ricerca
# reload: dopo una modifica a un file di src/ basta rieseguire questa cella, senza riavviare il kernel
for modulo in (wc_retrieval.grafo, wc_retrieval.ricerca):
    importlib.reload(modulo)
from wc_retrieval import ricerca
from wc_retrieval.grafo import Grafo

# Connessione a Neo4j: la password si chiede e non si salva
g = Grafo(getpass("Password Neo4j: "))

print("codice:", SRC, "| connessione:", g.leggi("CALL dbms.components() YIELD versions RETURN versions[0] AS v")["v"][0])

codice: <progetto>\src | connessione: 5.26.19


In [ ]:
# [Cella 2] Indici full-text (SCRIVE: solo indici): si aggiungono allenatori e arbitri
# Crea i sei indici Lucene sui nomi: giocatori, federazioni, club, stadi, allenatori, arbitri
ricerca.crea_indici_fulltext(g)                          # IF NOT EXISTS: crea solo quelli che mancano
print(g.leggi("""SHOW FULLTEXT INDEXES YIELD name, state, labelsOrTypes, properties
                 RETURN name AS indice, state AS stato, labelsOrTypes AS classe, properties AS proprieta
                 ORDER BY indice""").to_string(index=False))
# Due prove: nomi scritti male trovati comunque (ricerca fuzzy)
print(ricerca.cerca_nome(g, "Manager", "Marcelo Lipi", k=3)[["nome", "dettaglio", "punteggio"]].to_string(index=False))
print(ricerca.cerca_nome(g, "Referee", "Colina", k=3)[["nome", "dettaglio", "punteggio"]].to_string(index=False))

          indice  stato       classe         proprieta
 nomi_allenatori ONLINE    [Manager]           [label]
    nomi_arbitri ONLINE    [Referee]           [label]
       nomi_club ONLINE       [Club]           [label]
nomi_federazioni ONLINE [Federation] [label, altLabel]
  nomi_giocatori ONLINE     [Player]           [label]
      nomi_stadi ONLINE    [Stadium] [label, cityName]
          nome                dettaglio  punteggio
Marcello Lippi [Italy 2006, Italy 2010]   4.097306
             nome dettaglio  punteggio
Pierluigi Collina   [Italy]    2.29547
